## Example 1: Multi-provider LLM calls with unified API

In [ ]:
import os
os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")
from litellm import completion
print("✅ LiteLLM imported successfully")

# Example 1: Multi-provider LLM calls with unified API
print("Example 1: Multi-Provider LLM Calls with Unified OpenAI Format")

# The provider/ prefix names the provider explicitly, so routing does not depend on litellm's model list
models_to_test = [
    "openai/gpt-3.5-turbo",
    "anthropic/claude-3-haiku-20240307",
    "cohere/command-nightly",
]
message = [{"content": "Explain quantum computing in one sentence.", "role": "user"}]

# mock_response= short-circuits the provider call, so the unified completion() path runs offline and returns the same
# OpenAI-format ModelResponse for every provider
print("\n🔄 Unified API: the same call for every provider")
for model in models_to_test:
    response = completion(model=model, messages=message, mock_response="Qubits exploit superposition.")
    content = response.choices[0].message.content
    print(f"  • {model}: {content!r} ({response.usage.total_tokens} tokens)")
    assert content == "Qubits exploit superposition.", (model, content)
    assert response.model == model.split("/", 1)[1], (model, response.model)  # the response names the model without its prefix

print("Note: real calls need provider API keys; mock_response exercises the same code path offline")
print("Example 1: Multi-provider LLM calls completed")

## Example 2: Cost tracking and budget management

In [ ]:
import math
import litellm
from litellm import completion, cost_per_token, completion_cost, model_cost
print("✅ LiteLLM cost tracking imported successfully")

# Example 2: Cost tracking and budget management
print("Example 2: Cost Tracking and Budget Management")

# The budget is the module attribute litellm.max_budget.
previous_budget = litellm.max_budget
litellm.max_budget = 100.0  # $100 USD
print(f"💰 Budget set to: ${litellm.max_budget}")

# cost_per_token prices a call from the model cost map
input_tokens, output_tokens, model = 50, 150, "gpt-3.5-turbo"
prompt_cost, output_cost = cost_per_token(model=model, prompt_tokens=input_tokens, completion_tokens=output_tokens)
prices = model_cost[model]
print(f"\n📊 Cost Calculation Example: {model}, {input_tokens} input / {output_tokens} output tokens")
print(f"  Prompt cost: ${prompt_cost:.6f}   Completion cost: ${output_cost:.6f}")
assert math.isclose(prompt_cost, input_tokens * prices["input_cost_per_token"]), prompt_cost
assert math.isclose(output_cost, output_tokens * prices["output_cost_per_token"]), output_cost

# completion_cost prices a response from its usage block (a mocked response carries one)
response = completion(model=model, messages=[{"role": "user", "content": "hi"}], mock_response="hello")
cost = completion_cost(completion_response=response)
expected = (response.usage.prompt_tokens * prices["input_cost_per_token"]
            + response.usage.completion_tokens * prices["output_cost_per_token"])
print(f"  Mocked call: {response.usage.total_tokens} tokens -> ${cost:.6f}")
assert math.isclose(cost, expected), (cost, expected)
assert cost * 1000 < litellm.max_budget, cost  # a thousand such calls fit the budget

litellm.max_budget = previous_budget
print("Example 2: Cost tracking and budgets demonstrated")

## Example 3: Completion with Fallbacks and Retry Logic

In [ ]:
# Example 3: Completion with Fallbacks and Retry Logic
print("=" * 70)
print(f"🚀 Example 3: LiteLLM Fallbacks and Retry Logic")
print("=" * 70)

import litellm
from litellm import Router
print("✅ LiteLLM fallback features imported successfully")

messages = [{"content": "Write a haiku about machine learning", "role": "user"}]
litellm.suppress_debug_info = True  # keep litellm's "Give Feedback / Get Help" banner out of the expected-error demo

# A provider failure surfaces as litellm's mapped exception type: mock_response=Exception fails the call offline, and a
# "rate limited" error maps to litellm.RateLimitError — the class retries and fallbacks key on
try:
    litellm.completion(model="gpt-4-turbo", messages=messages, mock_response=Exception("rate limited"), num_retries=0)
except litellm.RateLimitError as e:
    assert "rate limited" in str(e), str(e)
    print(f"  Primary call rejected as expected: {type(e).__name__}")
else:
    raise AssertionError("a mocked provider error did not raise")

# Router fallbacks: the primary deployment fails and the router answers from the configured fallback group
router = Router(
    model_list=[
        {"model_name": "primary", "litellm_params": {"model": "openai/gpt-4-turbo", "api_key": "sk-demo",
                                                     "mock_response": Exception("primary is down")}},
        {"model_name": "backup", "litellm_params": {"model": "openai/gpt-3.5-turbo", "api_key": "sk-demo",
                                                    "mock_response": "Gradients descend"}},
    ],
    fallbacks=[{"primary": ["backup"]}],
    num_retries=0,
)
response = router.completion(model="primary", messages=messages)
print(f"\n🔄 Fallback answered {response.choices[0].message.content!r} via {response.model}")
assert response.choices[0].message.content == "Gradients descend", response.choices[0].message.content
assert response.model == "gpt-3.5-turbo", response.model

print("Example 3: Fallbacks and retry logic demonstrated")

## Example 4: Token Counting and Usage Analytics

In [ ]:
# Example 4: Token Counting and Usage Analytics
print("=" * 70)
print(f"🚀 Example 4: LiteLLM Token Counting and Usage Analytics")
print("=" * 70)

# Token counting uses litellm.token_counter.
from litellm import token_counter, model_cost
print("✅ LiteLLM token counting imported successfully")

sample_messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "Explain the benefits of renewable energy in detail."},
    {"role": "assistant", "content": "Renewable energy offers numerous benefits including environmental sustainability, cost savings, energy independence, and job creation."}
]

print("\n🔢 Token Count Analysis:")
counts = {}
for model in ["gpt-4", "gpt-3.5-turbo"]:
    counts[model] = token_counter(model=model, messages=sample_messages)
    print(f"  • {model}: {counts[model]} tokens (context window {model_cost[model]['max_input_tokens']})")
    assert 0 < counts[model] < model_cost[model]["max_input_tokens"], counts[model]

# both models use tiktoken's cl100k_base, which litellm bundles, so the counts agree and nothing is downloaded
assert counts["gpt-4"] == counts["gpt-3.5-turbo"], counts
assert token_counter(model="gpt-4", text="hello world") == 2

print("Example 4: Token counting and usage analytics demonstrated")

## Example 5: Router and Load Balancing

In [ ]:
# Example 5: Router and Load Balancing
print("=" * 70)
print(f"🚀 Example 5: LiteLLM Router and Load Balancing")
print("=" * 70)

import litellm
from litellm import Router
print("✅ LiteLLM Router imported successfully")

# Example 5: Router and load balancing across providers; mock_response keeps every deployment offline
model_list = [
    {
        "model_name": "gpt-4",
        "litellm_params": {"model": "azure/gpt-4", "api_key": "azure-key", "mock_response": "from azure"},
        "tpm": 240000,  # tokens per minute
        "rpm": 1800     # requests per minute
    },
    {
        "model_name": "gpt-4",
        "litellm_params": {"model": "openai/gpt-4", "api_key": "openai-key", "mock_response": "from openai"},
        "tpm": 300000,
        "rpm": 5000
    },
    {
        "model_name": "claude-3",
        "litellm_params": {"model": "anthropic/claude-3-sonnet", "api_key": "anthropic-key", "mock_response": "from anthropic"},
        "tpm": 200000,
        "rpm": 1000
    }
]
router = Router(model_list=model_list)
print(f"✅ Router configured: groups {sorted(router.model_names)} over {len(router.model_list)} deployments")
assert sorted(router.model_names) == ["claude-3", "gpt-4"], router.model_names
assert len(router.get_model_list(model_name="gpt-4")) == 2

messages = [{"role": "user", "content": "Describe load balancing in one line."}]
answers = [router.completion(model="gpt-4", messages=messages).choices[0].message.content for _ in range(4)]
print(f"\n⚖️ gpt-4 group answered from: {answers}")
assert set(answers) <= {"from azure", "from openai"}, answers  # each call lands on one of the group's two deployments
claude = router.completion(model="claude-3", messages=messages).choices[0].message.content
print(f"  claude-3 group answered: {claude!r}")
assert claude == "from anthropic", claude

# an unknown model group is refused with BadRequestError rather than routed anywhere
try:
    router.completion(model="gpt-5-unknown", messages=messages)
except litellm.BadRequestError as e:
    assert "no healthy deployments" in str(e), str(e)
    print("  Unknown model group rejected as expected")
else:
    raise AssertionError("the router accepted an unknown model group")

print("Example 5: Router and load balancing demonstrated")